In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv


In [2]:
import re

df = pd.read_csv("/kaggle/input/datasets/kazanova/sentiment140/training.1600000.processed.noemoticon.csv", encoding="latin-1", header=None)
df.columns = ["label","id","date","query","user","text"]

def clean(text):
    text = re.sub(r"http\S+", "", text)       # remove URLs
    text = re.sub(r"@\w+", "", text)          # remove handles
    text = re.sub(r"[^A-Za-z0-9\s!?.,']", "", text)
    return text.strip()

df["text"] = df["text"].apply(clean)
df["label"] = df["label"].apply(lambda x: 1 if x == 4 else 0)  # binary: 0/1

# Sample 50k rows to keep training fast
df = df.sample(50000, random_state=42)

In [3]:
df.head()

,label,id,date,query,user,text
541200,0,2200003196,Tue Jun 16 18:18:12 PDT 2009,NO_QUERY,LaLaLindsey0609,AHHH I HOPE YOUR OK!!!
750,0,1467998485,Mon Apr 06 23:11:14 PDT 2009,NO_QUERY,sexygrneyes,"cool , i have no tweet apps for my razr 2"
766711,0,2300048954,Tue Jun 23 13:40:11 PDT 2009,NO_QUERY,sammydearr,i know just family drama. its lame.hey next t...
285055,0,1993474027,Mon Jun 01 10:26:07 PDT 2009,NO_QUERY,Lamb_Leanne,School email won't open and I have geography ...
705995,0,2256550904,Sat Jun 20 12:56:51 PDT 2009,NO_QUERY,yogicerdito,upper airways problem


In [4]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

dataset = Dataset.from_pandas(df[["text","label"]])
dataset = dataset.map(tokenize, batched=True)
dataset = dataset.train_test_split(test_size=0.1)

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(model=model, args=args, train_dataset=dataset["train"], eval_dataset=dataset["test"])
trainer.train()
model.save_pretrained("./tweet-classifier")
tokenizer.save_pretrained("./tweet-classifier")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./tweet-classifier/tokenizer_config.json',
 './tweet-classifier/tokenizer.json')

In [5]:
!touch .env

/bin/bash: line 1: nano: command not found


In [9]:
!echo "HF_TOKEN=\"***********************\"" >> .env

In [13]:
from huggingface_hub import HfApi
from dotenv import load_dotenv
import os

load_dotenv()

api = HfApi(token=os.getenv("HF_TOKEN"))

folder = "./tweet-classifier"
repo_id = "KinSlay3rs/tweet-tone-classifier"

# get all files in the folder
files = os.listdir(folder)
print(f"Files to upload: {files}")

for filename in files:
    filepath = os.path.join(folder, filename)
    print(f"Uploading {filename}...")
    
    api.upload_file(
        path_or_fileobj=filepath,
        path_in_repo=filename,
        repo_id=repo_id,
        repo_type="model",
        token=os.getenv("HF_TOKEN")
    )
    
    print(f"✅ {filename} uploaded!")

print("\n🎉 All files uploaded successfully!")

Files to upload: ['tokenizer_config.json', 'tokenizer.json', 'config.json', 'model.safetensors']
Uploading tokenizer_config.json...
✅ tokenizer_config.json uploaded!
Uploading tokenizer.json...
✅ tokenizer.json uploaded!
Uploading config.json...
✅ config.json uploaded!
Uploading model.safetensors...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ model.safetensors uploaded!

🎉 All files uploaded successfully!
